# 交易明细导入 MySQL

将已核验的 10 月有效明细导入 data_10month 库的 lavel2 表。

In [1]:
import pandas as pd
from sqlalchemy import URL, create_engine, text

from mysql_import import write_dataframe_to_mysql

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / "../config/.env", interpolate=False, encoding="utf-8-sig")
mysql_user = os.getenv("UCI_MYSQL_USER")
mysql_password = os.getenv("UCI_MYSQL_PASSWORD")
if not mysql_user or not mysql_password:
    raise RuntimeError("Set UCI_MYSQL_USER and UCI_MYSQL_PASSWORD in the kernel environment or in ../config/.env.")

# 创建数据库连接地址
url = URL.create(
    drivername="mysql+pymysql",
    username=mysql_user,
    password=mysql_password,
    host="127.0.0.1",
    port=3306,
    database="data_10month",
    query={"charset": "utf8mb4"}
)

# 创建连接引擎
engine = create_engine(url, pool_pre_ping=True)

# 检查连接的数据库和 MySQL 版本
with engine.connect() as conn:
    result = conn.execute(
        text("SELECT DATABASE(), VERSION()")
    )
    print(result.fetchone())

('data_10month', '8.4.11')


In [3]:
df = pd.read_csv("../data/processed/data_10month.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57834 entries, 0 to 57833
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   invoiceno    57834 non-null  int64  
 1   stockcode    57834 non-null  object 
 2   description  57834 non-null  object 
 3   quantity     57834 non-null  int64  
 4   invoicedate  57834 non-null  object 
 5   unitprice    57834 non-null  float64
 6   customerid   48096 non-null  float64
 7   country      57834 non-null  object 
 8   lineamount   57834 non-null  float64
dtypes: float64(3), int64(2), object(4)
memory usage: 4.0+ MB


In [ ]:
write_dataframe_to_mysql(df, "lavel2", engine)

57834